# Lazada 쇼핑몰 SQLite DB + Gradio 대시보드 + Text-to-SQL 챗봇

이 노트북은 업로드한 쇼핑몰 CSV를 기준으로 다음을 수행합니다.

1. SQLite DB 파일 생성
2. ERD 기준 테이블 생성 및 데이터 적재
3. Gradio 인터랙티브 대시보드 제작
4. OpenAI API 기반 Text-to-SQL 챗봇 구현

권장 입력 파일:
- `lazada_products_merged_by_category_id.csv`
- 또는 `lazada_products_final_preprocessed.csv` + `categories.csv`

In [1]:
# STEP 0. 패키지 설치
!pip install -q gradio plotly openai pandas

In [2]:
# STEP 0-1. 라이브러리 로드
import os
import re
import sqlite3
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import gradio as gr

DB_PATH = 'lazada_shoppingmall.db'
PRODUCT_CSV = 'lazada_products_merged_by_category_id.csv'
CATEGORY_CSV = 'categories.csv'

print('준비 완료')

준비 완료


In [3]:
# STEP 0-2. CSV 업로드
# Colab에서 실행 시, 파일이 없으면 업로드 창이 뜹니다.

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

needed_files = [PRODUCT_CSV]
missing = [f for f in needed_files if not Path(f).exists()]

if missing and IN_COLAB:
    print('상품 CSV 파일을 업로드하세요. 예: lazada_products_merged_by_category_id.csv')
    uploaded = files.upload()
    print('업로드 완료:', list(uploaded.keys()))
else:
    print('파일 확인 완료')

# 업로드 파일명이 다를 경우 자동 탐색
csv_files = list(Path('.').glob('*.csv'))
print('현재 CSV 파일 목록:', [str(f) for f in csv_files])

if not Path(PRODUCT_CSV).exists():
    candidates = [f for f in csv_files if 'merged' in f.name.lower() or 'preprocessed' in f.name.lower() or 'lazada' in f.name.lower()]
    if candidates:
        PRODUCT_CSV = str(candidates[0])
        print('자동 선택된 상품 CSV:', PRODUCT_CSV)
    else:
        raise FileNotFoundError('상품 CSV 파일을 찾을 수 없습니다.')

if not Path(CATEGORY_CSV).exists():
    cat_candidates = [f for f in csv_files if 'categor' in f.name.lower()]
    if cat_candidates:
        CATEGORY_CSV = str(cat_candidates[0])
        print('자동 선택된 카테고리 CSV:', CATEGORY_CSV)
    else:
        CATEGORY_CSV = None
        print('별도 categories.csv 없음: 상품 CSV 안의 카테고리 컬럼으로 생성합니다.')

상품 CSV 파일을 업로드하세요. 예: lazada_products_merged_by_category_id.csv


Saving lazada_products.csv to lazada_products.csv
업로드 완료: ['lazada_products.csv']
현재 CSV 파일 목록: ['lazada_products.csv']
자동 선택된 상품 CSV: lazada_products.csv
별도 categories.csv 없음: 상품 CSV 안의 카테고리 컬럼으로 생성합니다.


In [4]:
# STEP 0-3. 데이터 로드 및 기본 확인
raw = pd.read_csv(PRODUCT_CSV)
raw.columns = [c.strip() for c in raw.columns]

print('상품 CSV:', PRODUCT_CSV)
print('행/열:', raw.shape)
display(raw.head())
print(raw.columns.tolist())

상품 CSV: lazada_products.csv
행/열: (945, 37)


,product_id,url,title,rating,reviews,initial_price,final_price,currency,image_url,seller_name,...,number_sold,gmv,seller_id,category_path,category_name,category_id,category_name_cat,parent_category_id,level,category_path_cat
0,1,https://www.lazada.co.id/products/dioda-damper...,dioda damper dmv 1500 tv polytron new original...,4.6,27,0.56,0.56,usd,"[""https://img.lazcdn.com/g/ff/kf/sc92f4b3c99c0...",yudiana yudi sperpat elektronik,...,111,62.15,1,"[""televisi & video"", ""televisi digital""]",televisi digital,2,Televisi Digital,1.0,2,"[""Televisi & Video"", ""Televisi Digital""]"
1,2,https://www.lazada.co.id/products/victus-lapto...,victus laptop gaming hp amd ryzen 5 nvidia gef...,NaN,0,982.89,923.77,usd,"[""https://img.lazcdn.com/g/p/3440632c069eddd82...",hp,...,0,0.00,2,"[""komputer & laptop"", ""laptop"", ""laptop umum""]",laptop umum,5,Laptop Umum,4.0,3,"[""Komputer & Laptop"", ""Laptop"", ""Laptop Umum""]"
2,3,https://www.lazada.co.id/products/laptop-hp-15...,laptop hp 15s-fq5148tu core i3 uhd 4gb & 8gb r...,5.0,47,363.87,341.48,usd,"[""https://img.lazcdn.com/g/p/23782d5f1bd37a89c...",hp,...,91,31074.51,2,"[""komputer & laptop"", ""laptop"", ""laptop umum""]",laptop umum,5,Laptop Umum,4.0,3,"[""Komputer & Laptop"", ""Laptop"", ""Laptop Umum""]"
3,4,https://www.lazada.co.id/products/printer-hp-d...,printer hp deskjet 2336 all in one ( print sca...,5.0,177,55.43,49.27,usd,"[""https://img.lazcdn.com/g/p/bf68d11f46c85761a...",hp,...,409,20151.63,2,"[""pencetak & monitor"", ""printer"", ""printer ink...",printer ink jet,8,Printer Ink Jet,7.0,3,"[""Pencetak & Monitor"", ""Printer"", ""Printer Ink..."
4,5,https://www.lazada.co.id/products/roda-koper-r...,"roda koper, roda pengganti, roda double wheel,...",5.0,2,3.64,3.64,usd,"[""https://img.lazcdn.com/g/p/abc4f89073ba5380c...",dewi05588,...,2,7.28,3,"[""tas & travel"", ""tas anak"", ""koper""]",koper,11,Koper,10.0,3,"[""Tas & Travel"", ""Tas Anak"", ""Koper""]"


['product_id', 'url', 'title', 'rating', 'reviews', 'initial_price', 'final_price', 'currency', 'image_url', 'seller_name', 'product_specifications', 'product_description', 'seller_ratings', 'seller_ship_on_time', 'seller_chat_response', 'sku', 'mpn', 'colors', 'variations', 'color', 'returns_and_warranty', 'is_super_seller', 'promotions', 'brand', 'product_variation', 'lazmall', 'domain', 'number_sold', 'gmv', 'seller_id', 'category_path', 'category_name', 'category_id', 'category_name_cat', 'parent_category_id', 'level', 'category_path_cat']


## STEP 1. SQLite DB 파일 만들기

아래 셀은 기존 DB 파일이 있으면 삭제하고 새로 생성합니다.

In [5]:
# STEP 1. SQLite DB 파일 생성
if Path(DB_PATH).exists():
    Path(DB_PATH).unlink()

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute('PRAGMA foreign_keys = ON;')
print(f'SQLite DB 생성 완료: {DB_PATH}')

SQLite DB 생성 완료: lazada_shoppingmall.db


## STEP 2. ERD 설계대로 테이블 구성

사용 테이블:
- `sellers`
- `brands`
- `categories`
- `products`
- `product_prices`
- `product_sales_stats`
- `product_images`

In [33]:
# STEP 2-1. 전처리 함수 및 No Brand 보정

def to_numeric(series):
    return pd.to_numeric(series, errors='coerce')

def clean_bool_to_int(series):
    return series.astype(str).str.lower().map({
        'true': 1, 'false': 0, '1': 1, '0': 0, 'yes': 1, 'no': 0
    }).fillna(0).astype(int)

def ensure_col(df, col, default=np.nan):
    if col not in df.columns:
        df[col] = default
    return df

# 필수 컬럼 보정
for col in ['product_id', 'seller_id', 'category_id']:
    if col not in raw.columns:
        raw[col] = range(1, len(raw) + 1)

# 숫자형 컬럼 정리
num_cols = [
    'rating', 'reviews', 'initial_price', 'final_price', 'seller_ratings',
    'seller_ship_on_time', 'seller_chat_response', 'number_sold', 'gmv',
    'product_id', 'seller_id', 'category_id', 'parent_category_id', 'level'
]
for col in num_cols:
    if col in raw.columns:
        raw[col] = to_numeric(raw[col])

# boolean 컬럼 정리
for col in ['is_super_seller', 'lazmall']:
    if col in raw.columns:
        raw[col] = clean_bool_to_int(raw[col])

# brand 보정: 'no brand'인 경우 title의 첫 단어 사용
if 'brand' in raw.columns:
    raw['brand'] = raw['brand'].fillna('no brand').astype(str)
    mask = raw['brand'].str.lower().str.strip() == 'no brand'
    # title에서 첫 단어 추출 (공백 기준)
    raw.loc[mask, 'brand'] = raw.loc[mask, 'title'].fillna('unknown').str.split().str[0]

# brand_id 생성
brand_map = {name: idx for idx, name in enumerate(sorted(raw['brand'].unique()), start=1)}
raw['brand_id'] = raw['brand'].map(brand_map).astype(int)

print('전처리 및 브랜드 보정 완료')

전처리 및 브랜드 보정 완료


In [7]:
# STEP 2-2. 정규화 테이블 DataFrame 생성

# sellers
seller_cols = ['seller_id', 'seller_name', 'seller_ratings', 'seller_ship_on_time', 'seller_chat_response', 'is_super_seller']
for c in seller_cols:
    ensure_col(raw, c)
sellers_df = raw[seller_cols].drop_duplicates('seller_id').sort_values('seller_id')
sellers_df['seller_name'] = sellers_df['seller_name'].fillna('unknown')

# brands
brands_df = pd.DataFrame({
    'brand_id': list(brand_map.values()),
    'brand_name': list(brand_map.keys())
}).sort_values('brand_id')

# categories
if CATEGORY_CSV and Path(CATEGORY_CSV).exists():
    categories_df = pd.read_csv(CATEGORY_CSV)
    categories_df.columns = [c.strip() for c in categories_df.columns]
else:
    cat_cols = ['category_id', 'category_name', 'parent_category_id', 'level', 'category_path']
    for c in cat_cols:
        ensure_col(raw, c)
    categories_df = raw[cat_cols].drop_duplicates('category_id').copy()

# 병합 파일의 *_cat 컬럼 우선 사용 가능
if 'category_name_cat' in raw.columns and 'category_name' not in categories_df.columns:
    categories_df['category_name'] = raw['category_name_cat']
if 'category_path_cat' in raw.columns and 'category_path' not in categories_df.columns:
    categories_df['category_path'] = raw['category_path_cat']

for c in ['category_id', 'category_name', 'parent_category_id', 'level', 'category_path']:
    ensure_col(categories_df, c)

categories_df = categories_df[['category_id', 'category_name', 'parent_category_id', 'level', 'category_path']].drop_duplicates('category_id')
categories_df['category_id'] = pd.to_numeric(categories_df['category_id'], errors='coerce').astype('Int64')
categories_df = categories_df.dropna(subset=['category_id'])
categories_df['category_id'] = categories_df['category_id'].astype(int)
categories_df['parent_category_id'] = pd.to_numeric(categories_df['parent_category_id'], errors='coerce').astype('Int64')
categories_df['level'] = pd.to_numeric(categories_df['level'], errors='coerce').fillna(1).astype(int)
categories_df['category_name'] = categories_df['category_name'].fillna('unknown')
categories_df['category_path'] = categories_df['category_path'].fillna(categories_df['category_name'])

# products
product_cols = [
    'product_id', 'seller_id', 'brand_id', 'category_id', 'title', 'sku', 'mpn',
    'product_specifications', 'product_description', 'colors', 'variations', 'color',
    'product_variation', 'returns_and_warranty', 'lazmall', 'domain', 'url'
]
for c in product_cols:
    ensure_col(raw, c)
products_df = raw[product_cols].drop_duplicates('product_id').copy()
products_df['title'] = products_df['title'].fillna('unknown')
products_df['seller_id'] = products_df['seller_id'].astype(int)
products_df['brand_id'] = products_df['brand_id'].astype(int)
products_df['category_id'] = pd.to_numeric(products_df['category_id'], errors='coerce').astype('Int64')
products_df['lazmall'] = products_df['lazmall'].fillna(0).astype(int)

# prices
price_cols = ['product_id', 'initial_price', 'final_price', 'currency', 'promotions']
for c in price_cols:
    ensure_col(raw, c)
product_prices_df = raw[price_cols].drop_duplicates('product_id').copy()
product_prices_df.insert(0, 'price_id', range(1, len(product_prices_df) + 1))
product_prices_df['currency'] = product_prices_df['currency'].fillna('usd')

# sales stats
stat_cols = ['product_id', 'rating', 'reviews', 'number_sold', 'gmv']
for c in stat_cols:
    ensure_col(raw, c)
product_sales_stats_df = raw[stat_cols].drop_duplicates('product_id').copy()
product_sales_stats_df.insert(0, 'stat_id', range(1, len(product_sales_stats_df) + 1))

# images
if 'image_url' not in raw.columns and 'image' in raw.columns:
    raw['image_url'] = raw['image']
ensure_col(raw, 'image_url')
product_images_df = raw[['product_id', 'image_url']].drop_duplicates('product_id').copy()
product_images_df.insert(0, 'image_id', range(1, len(product_images_df) + 1))
product_images_df['sort_order'] = 1

print('정규화 테이블 생성 완료')
for name, df in {
    'sellers': sellers_df,
    'brands': brands_df,
    'categories': categories_df,
    'products': products_df,
    'product_prices': product_prices_df,
    'product_sales_stats': product_sales_stats_df,
    'product_images': product_images_df,
}.items():
    print(name, df.shape)

정규화 테이블 생성 완료
sellers (196, 6)
brands (78, 2)
categories (79, 5)
products (945, 17)
product_prices (945, 6)
product_sales_stats (945, 6)
product_images (945, 4)


In [8]:
# STEP 2-3. SQLite 테이블 생성
cur.executescript('''
DROP TABLE IF EXISTS product_images;
DROP TABLE IF EXISTS product_sales_stats;
DROP TABLE IF EXISTS product_prices;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS categories;
DROP TABLE IF EXISTS brands;
DROP TABLE IF EXISTS sellers;

CREATE TABLE sellers (
    seller_id INTEGER PRIMARY KEY,
    seller_name TEXT NOT NULL,
    seller_ratings REAL,
    seller_ship_on_time REAL,
    seller_chat_response REAL,
    is_super_seller INTEGER NOT NULL DEFAULT 0 CHECK (is_super_seller IN (0, 1))
);

CREATE TABLE brands (
    brand_id INTEGER PRIMARY KEY,
    brand_name TEXT NOT NULL UNIQUE
);

CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT NOT NULL,
    parent_category_id INTEGER NULL,
    level INTEGER NOT NULL,
    category_path TEXT NOT NULL,
    FOREIGN KEY (parent_category_id) REFERENCES categories(category_id)
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    seller_id INTEGER NOT NULL,
    brand_id INTEGER NOT NULL,
    category_id INTEGER,
    title TEXT NOT NULL,
    sku TEXT,
    mpn TEXT,
    product_specifications TEXT,
    product_description TEXT,
    colors TEXT,
    variations TEXT,
    color TEXT,
    product_variation TEXT,
    returns_and_warranty TEXT,
    lazmall INTEGER NOT NULL DEFAULT 0 CHECK (lazmall IN (0, 1)),
    domain TEXT,
    url TEXT UNIQUE,
    FOREIGN KEY (seller_id) REFERENCES sellers(seller_id),
    FOREIGN KEY (brand_id) REFERENCES brands(brand_id),
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
);

CREATE TABLE product_prices (
    price_id INTEGER PRIMARY KEY,
    product_id INTEGER NOT NULL UNIQUE,
    initial_price REAL,
    final_price REAL NOT NULL,
    currency TEXT,
    promotions TEXT,
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

CREATE TABLE product_sales_stats (
    stat_id INTEGER PRIMARY KEY,
    product_id INTEGER NOT NULL UNIQUE,
    rating REAL,
    reviews INTEGER,
    number_sold INTEGER,
    gmv REAL,
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

CREATE TABLE product_images (
    image_id INTEGER PRIMARY KEY,
    product_id INTEGER NOT NULL,
    image_url TEXT,
    sort_order INTEGER DEFAULT 1,
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
''')
conn.commit()
print('테이블 생성 완료')

테이블 생성 완료


In [34]:
# 데이터 보정 후 DB 재적재
sellers_df.to_sql('sellers', conn, if_exists='replace', index=False)

# 보정된 brand_map으로 brands_df 업데이트 후 적재
brands_df = pd.DataFrame({
    'brand_id': list(brand_map.values()),
    'brand_name': list(brand_map.keys())
}).sort_values('brand_id')
brands_df.to_sql('brands', conn, if_exists='replace', index=False)

categories_df.to_sql('categories', conn, if_exists='replace', index=False)

# 보정된 brand_id가 포함된 products_df 업데이트 후 적재
products_df = raw[product_cols].drop_duplicates('product_id').copy()
products_df.to_sql('products', conn, if_exists='replace', index=False)

product_prices_df.to_sql('product_prices', conn, if_exists='replace', index=False)
product_sales_stats_df.to_sql('product_sales_stats', conn, if_exists='replace', index=False)
product_images_df.to_sql('product_images', conn, if_exists='replace', index=False)

conn.commit()

print('보정된 데이터로 DB 업데이트 완료')

보정된 데이터로 DB 업데이트 완료


In [15]:
# STEP 2-5. 검증용 SQL
validation_queries = {
    '상품-셀러 FK 누락': '''
        SELECT COUNT(*) AS cnt
        FROM products p
        LEFT JOIN sellers s ON p.seller_id = s.seller_id
        WHERE s.seller_id IS NULL;
    ''',
    '상품-카테고리 FK 누락': '''
        SELECT COUNT(*) AS cnt
        FROM products p
        LEFT JOIN categories c ON p.category_id = c.category_id
        WHERE p.category_id IS NOT NULL AND c.category_id IS NULL;
    ''',
    '상품-가격 누락': '''
        SELECT COUNT(*) AS cnt
        FROM products p
        LEFT JOIN product_prices pp ON p.product_id = pp.product_id
        WHERE pp.product_id IS NULL;
    '''
}

for name, q in validation_queries.items():
    result = pd.read_sql(q, conn)
    print(name, ':', result.iloc[0, 0])

상품-셀러 FK 누락 : 0
상품-카테고리 FK 누락 : 0
상품-가격 누락 : 0


## STEP 3. Gradio 인터랙티브 대시보드 제작

구성:
1. KPI 카드 4개
2. 카테고리별 매출 TOP10
3. 슈퍼셀러 vs 일반셀러 비교
4. 평점 분포 히스토그램
5. 베스트셀러 TOP10
6. 카테고리 내 TOP3, 윈도우 함수 RANK 활용

In [35]:
import plotly.subplots as sp
import plotly.graph_objects as go

# STEP 3-1. 대시보드 SQL 함수

def run_sql(sql, params=None):
    with sqlite3.connect(DB_PATH) as c:
        return pd.read_sql(sql, c, params=params or {})

def get_kpis():
    sql = '''
    SELECT
        COUNT(DISTINCT p.product_id) AS 총상품수,
        COUNT(DISTINCT s.seller_id) AS 총셀러수,
        COALESCE(SUM(ps.gmv), 0) AS 총GMV,
        AVG(ps.rating) AS 평균평점
    FROM products p
    JOIN sellers s ON p.seller_id = s.seller_id
    LEFT JOIN product_sales_stats ps ON p.product_id = ps.product_id;
    '''
    df = run_sql(sql)
    row = df.iloc[0]
    return (
        f"{int(row['총상품수']):,}",
        f"{int(row['총셀러수']):,}",
        f"${row['총GMV']:,.2f}",
        f"{row['평균평점']:.2f}" if pd.notna(row['평균평점']) else 'N/A'
    )

def category_sales_top10():
    sql = '''
    SELECT
        c.category_name AS 카테고리,
        SUM(ps.gmv) AS 총매출
    FROM products p
    JOIN categories c ON p.category_id = c.category_id
    JOIN product_sales_stats ps ON p.product_id = ps.product_id
    GROUP BY c.category_name
    ORDER BY SUM(ps.gmv) DESC
    LIMIT 10;
    '''
    df = run_sql(sql)
    fig = px.bar(df, x='총매출', y='카테고리', orientation='h', title='카테고리별 매출 TOP10')
    fig.update_layout(yaxis={'categoryorder': 'total ascending'})
    return fig

def seller_type_compare():
    sql = '''
    SELECT
        CASE WHEN s.is_super_seller = 1 THEN '슈퍼셀러' ELSE '일반셀러' END AS 셀러유형,
        COUNT(DISTINCT p.product_id) AS 상품수,
        SUM(COALESCE(ps.number_sold, 0)) AS 총판매수량,
        AVG(ps.rating) AS 평균평점
    FROM products p
    JOIN sellers s ON p.seller_id = s.seller_id
    LEFT JOIN product_sales_stats ps ON p.product_id = ps.product_id
    GROUP BY s.is_super_seller;
    '''
    df = run_sql(sql)

    # 스케일 차이로 인해 3개의 서브플롯 생성
    fig = sp.make_subplots(rows=1, cols=3, subplot_titles=('상품수 비교', '총판매수량 비교', '평균평점 비교'))

    colors = ['#636EFA', '#EF553B']

    # 1. 상품수
    fig.add_trace(go.Bar(x=df['셀러유형'], y=df['상품수'], name='상품수', marker_color=colors[0]), row=1, col=1)
    # 2. 총판매수량
    fig.add_trace(go.Bar(x=df['셀러유형'], y=df['총판매수량'], name='총판매수량', marker_color=colors[1]), row=1, col=2)
    # 3. 평균평점
    fig.add_trace(go.Bar(x=df['셀러유형'], y=df['평균평점'], name='평균평점', marker_color='#00CC96'), row=1, col=3)

    fig.update_layout(height=400, title_text="슈퍼셀러 vs 일반셀러 상세 지표 비교", showlegend=False)
    return fig

def rating_histogram():
    sql = '''
    SELECT rating AS 평점
    FROM product_sales_stats
    WHERE rating IS NOT NULL AND rating BETWEEN 0 AND 5;
    '''
    df = run_sql(sql)
    fig = px.histogram(df, x='평점', nbins=10, title='평점 분포 히스토그램')
    fig.update_xaxes(range=[0, 5])
    return fig

def bestseller_top10():
    sql = '''
    SELECT
        p.title AS 상품명,
        s.seller_name AS 셀러,
        pp.final_price AS 가격,
        ps.number_sold AS 판매량,
        ps.rating AS 평점
    FROM products p
    JOIN sellers s ON p.seller_id = s.seller_id
    JOIN product_prices pp ON p.product_id = pp.product_id
    JOIN product_sales_stats ps ON p.product_id = ps.product_id
    ORDER BY ps.number_sold DESC, ps.gmv DESC
    LIMIT 10;
    '''
    return run_sql(sql)

def category_top3():
    sql = '''
    WITH ranked AS (
        SELECT
            c.category_name AS 카테고리,
            p.title AS 상품명,
            s.seller_name AS 셀러,
            pp.final_price AS 가격,
            ps.number_sold AS 판매량,
            ps.rating AS 평점,
            ps.gmv AS GMV,
            RANK() OVER (
                PARTITION BY c.category_id
                ORDER BY ps.gmv DESC, ps.number_sold DESC
            ) AS 순위
        FROM products p
        JOIN categories c ON p.category_id = c.category_id
        JOIN sellers s ON p.seller_id = s.seller_id
        JOIN product_prices pp ON p.product_id = pp.product_id
        JOIN product_sales_stats ps ON p.product_id = ps.product_id
    )
    SELECT 카테고리, 순위, 상품명, 셀러, 가격, 판매량, 평점, GMV
    FROM ranked
    WHERE 순위 <= 3
    ORDER BY 카테고리, 순위;
    '''
    return run_sql(sql)

print('대시보드 함수 준비 완료 (그래프 분리 적용)')

대시보드 함수 준비 완료 (그래프 분리 적용)


In [17]:
# STEP 3-2. Gradio 대시보드 실행

def refresh_dashboard():
    total_products, total_sellers, total_gmv, avg_rating = get_kpis()
    return (
        total_products,
        total_sellers,
        total_gmv,
        avg_rating,
        category_sales_top10(),
        seller_type_compare(),
        rating_histogram(),
        bestseller_top10(),
        category_top3()
    )

with gr.Blocks(title='Lazada 쇼핑몰 데이터 대시보드') as dashboard:
    gr.Markdown('# Lazada 쇼핑몰 데이터 대시보드')
    gr.Markdown('SQLite DB 기반 인터랙티브 대시보드입니다.')

    refresh_btn = gr.Button('대시보드 새로고침')

    with gr.Row():
        total_products_box = gr.Textbox(label='총 상품 수')
        total_sellers_box = gr.Textbox(label='총 셀러 수')
        total_gmv_box = gr.Textbox(label='총 GMV')
        avg_rating_box = gr.Textbox(label='평균 평점')

    with gr.Tab('차트'):
        category_chart = gr.Plot(label='카테고리별 매출 TOP10')
        seller_compare_chart = gr.Plot(label='슈퍼셀러 vs 일반셀러 비교')
        rating_hist_chart = gr.Plot(label='평점 분포 히스토그램')

    with gr.Tab('테이블'):
        bestseller_table = gr.Dataframe(label='베스트셀러 TOP10')
        category_top3_table = gr.Dataframe(label='카테고리 내 TOP3')

    outputs = [
        total_products_box, total_sellers_box, total_gmv_box, avg_rating_box,
        category_chart, seller_compare_chart, rating_hist_chart,
        bestseller_table, category_top3_table
    ]
    refresh_btn.click(fn=refresh_dashboard, outputs=outputs)
    dashboard.load(fn=refresh_dashboard, outputs=outputs)

# Colab에서는 share=True 권장
dashboard.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c04b36eae367372c2c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## STEP 4. Text-to-SQL AI 챗봇 구현

작동 흐름:

자연어 질문 → SQL 생성 → 안전 검증 → 실행 → 결과 → 자연어 요약

OpenAI API Key는 Colab Secrets 또는 환경변수 `OPENAI_API_KEY`에 저장하세요.

In [31]:
# STEP 4-1. OpenAI API 설정
import os
from openai import OpenAI
from google.colab import userdata

try:
    api_key = userdata.get('OPENAI_API_KEY')
    if api_key:
        client = OpenAI(api_key=api_key)
        print('OpenAI API Key 로드 완료')
    else:
        client = None
        print('OPENAI_API_KEY를 찾을 수 없습니다. Colab Secrets에 등록해 주세요.')
except Exception as e:
    print(f'설정 중 오류 발생: {e}')
    client = None

OpenAI API Key 로드 완료


In [27]:
schema_info = """
[테이블 스키마]
products (
    product_id INTEGER PRIMARY KEY,
    seller_id INTEGER FK -> sellers.seller_id,
    brand_id INTEGER FK -> brands.brand_id,
    category_id INTEGER FK -> categories.category_id,
    title TEXT,
    sku TEXT,
    mpn TEXT,
    product_specifications TEXT,
    product_description TEXT,
    colors TEXT,
    variations TEXT,
    color TEXT,
    product_variation TEXT,
    returns_and_warranty TEXT,
    lazmall INTEGER 0/1,
    domain TEXT,
    url TEXT
)

sellers (
    seller_id INTEGER PRIMARY KEY,
    seller_name TEXT,
    seller_ratings REAL,
    seller_ship_on_time REAL,
    seller_chat_response REAL,
    is_super_seller INTEGER 0/1
)

brands (
    brand_id INTEGER PRIMARY KEY,
    brand_name TEXT
)

categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT,
    parent_category_id INTEGER NULL,
    level INTEGER,
    category_path TEXT
)

product_prices (
    price_id INTEGER PRIMARY KEY,
    product_id INTEGER FK -> products.product_id,
    initial_price REAL,
    final_price REAL,
    currency TEXT,
    promotions TEXT
)

product_sales_stats (
    stat_id INTEGER PRIMARY KEY,
    product_id INTEGER FK -> products.product_id,
    rating REAL,
    reviews INTEGER,
    number_sold INTEGER,
    gmv REAL
)

product_images (
    image_id INTEGER PRIMARY KEY,
    product_id INTEGER FK -> products.product_id,
    image_url TEXT,
    sort_order INTEGER
)
"""

system_prompt = f"""
당신은 SQLite SQL 전문가입니다.

{schema_info}

[규칙]
1. SQLite 문법만 사용합니다.
2. SELECT 쿼리만 작성합니다. INSERT/UPDATE/DELETE/DROP/ALTER/CREATE 금지.
3. JOIN이 필요하면 INNER JOIN을 우선 사용합니다.
4. 결과 컬럼에 한국어 alias를 사용합니다. 예: SUM(gmv) AS 총매출
5. SQL 코드만 출력하고 설명, 주석, 마크다운 코드블록을 추가하지 않습니다.
6. 테이블에 없는 컬럼을 만들지 않습니다.
7. 가격은 product_prices.final_price를 사용합니다.
8. 상품 평점은 product_sales_stats.rating을 사용합니다.
9. 셀러 평점은 sellers.seller_ratings를 사용합니다.
10. 판매량은 product_sales_stats.number_sold를 사용합니다.
11. 매출은 product_sales_stats.gmv를 사용합니다.
"""

DANGEROUS = [
    'DROP', 'DELETE', 'UPDATE', 'INSERT', 'ALTER', 'TRUNCATE',
    'CREATE', 'GRANT', 'REVOKE', 'REPLACE', 'ATTACH', 'DETACH',
    'PRAGMA', 'VACUUM'
]

def clean_sql(sql: str) -> str:
    sql = sql.strip()
    sql = re.sub(r'^```sql', '', sql, flags=re.IGNORECASE).strip()
    sql = re.sub(r'^```', '', sql).strip()
    sql = re.sub(r'```$', '', sql).strip()
    return sql

def is_safe_sql(sql: str):
    sql = clean_sql(sql)
    upper = sql.upper().strip()

    if not upper.startswith('SELECT') and not upper.startswith('WITH'):
        return False, 'SELECT 또는 WITH로 시작하는 조회 쿼리만 허용됩니다.'

    for kw in DANGEROUS:
        if re.search(rf'\b{kw}\b', upper):
            return False, f'위험 키워드 차단: {kw}'

    # 여러 문장 실행 방지: 마지막 세미콜론 1개만 허용
    without_last_semicolon = upper[:-1] if upper.endswith(';') else upper
    if ';' in without_last_semicolon:
        return False, '여러 SQL 문장을 한 번에 실행할 수 없습니다.'

    return True, '안전한 SQL입니다.'

def add_limit(sql: str, limit: int = 100):
    sql = clean_sql(sql).rstrip(';')

    # Remove any existing LIMIT or OFFSET clauses, especially at the end of the query.
    # This regex handles ' LIMIT N' and ' LIMIT N OFFSET M' and even ' LIMIT N LIMIT M'
    # at the end of the SQL string.
    sql = re.sub(
        r'\s+(?:LIMIT|OFFSET)\s+\d+(?:\s*(?:LIMIT|OFFSET)\s+\d+)*\s*$',
        '',
        sql,
        flags=re.IGNORECASE
    ).strip()

    # Now, add our desired LIMIT
    sql += f' LIMIT {limit}'
    return sql + ';'

print('프롬프트와 가드레일 준비 완료')


프롬프트와 가드레일 준비 완료


In [32]:
# STEP 4-3. SQL 생성, 실행, 요약 함수 (OpenAI 버전)

def generate_sql(question: str) -> str:
    if client is None:
        raise ValueError('OpenAI API Key가 설정되지 않았습니다.')

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': question}
        ],
        temperature=0
    )
    return clean_sql(response.choices[0].message.content)

def execute_safe_sql(sql: str):
    safe, msg = is_safe_sql(sql)
    if not safe:
        raise ValueError(msg)

    limited_sql = add_limit(sql, limit=100)
    with sqlite3.connect(DB_PATH) as c:
        df = pd.read_sql(limited_sql, c)
    return limited_sql, df

def summarize_answer(question: str, sql: str, df: pd.DataFrame) -> str:
    if client is None:
        return 'OpenAI API Key가 없어 요약을 생성하지 못했습니다.'

    preview = df.head(20).to_markdown(index=False)
    prompt = f"""
당신은 데이터 분석 결과를 쉽게 설명하는 분석가입니다.
사용자 질문: {question}
실행 SQL: {sql}
결과 미리보기:
{preview}

위 결과를 한국어로 짧고 명확하게 요약하세요.
데이터가 비어 있으면 조건에 맞는 결과가 없다고 말하세요.
"""
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': '당신은 데이터 분석 결과를 쉽게 설명하는 분석가입니다.'},
            {'role': 'user', 'content': prompt}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content

def ask_db(question: str):
    try:
        sql = generate_sql(question)
        limited_sql, df = execute_safe_sql(sql)
        summary = summarize_answer(question, limited_sql, df)
        return limited_sql, df, summary
    except Exception as e:
        return '', pd.DataFrame(), f'오류: {e}'

print('Text-to-SQL 함수 (OpenAI) 준비 완료')

Text-to-SQL 함수 (OpenAI) 준비 완료


In [28]:
# STEP 4-4. Text-to-SQL 테스트
# 예시 질문:
# 슈퍼셀러 중 셀러 평점이 4.5 이상인 셀러 10명 알려줘
# 카테고리별 GMV TOP10 알려줘
# 판매량이 가장 많은 상품 10개 보여줘

question = '슈퍼셀러 중 셀러 평점이 4.5 이상인 셀러 10명 알려줘'
sql, result_df, summary = ask_db(question)
print('생성 SQL:')
print(sql)
print('\n요약:')
print(summary)
display(result_df)

생성 SQL:
SELECT seller_id AS 셀러ID, seller_name AS 셀러명, seller_ratings AS 셀러평점
FROM sellers
WHERE is_super_seller = 1 AND seller_ratings >= 4.5
ORDER BY seller_ratings DESC LIMIT 100;

요약:
조건에 맞는 슈퍼셀러 중 셀러 평점이 4.5 이상인 셀러가 없습니다.


,셀러ID,셀러명,셀러평점


In [22]:
# STEP 4-5. Gradio Text-to-SQL 챗봇 실행

def chatbot_fn(question):
    sql, df, summary = ask_db(question)
    return sql, df, summary

with gr.Blocks(title='Lazada Text-to-SQL AI 챗봇') as sql_chatbot:
    gr.Markdown('# Lazada Text-to-SQL AI 챗봇')
    gr.Markdown('자연어 질문을 SQLite SQL로 변환하고, 안전 검증 후 실행합니다.')

    question_input = gr.Textbox(
        label='질문',
        placeholder='예: 슈퍼셀러 중 셀러 평점이 4.5 이상인 셀러 10명 알려줘',
        lines=2
    )
    run_btn = gr.Button('질문 실행')
    sql_output = gr.Code(label='생성된 SQL', language='sql')
    df_output = gr.Dataframe(label='SQL 실행 결과')
    summary_output = gr.Textbox(label='자연어 요약', lines=5)

    run_btn.click(
        fn=chatbot_fn,
        inputs=question_input,
        outputs=[sql_output, df_output, summary_output]
    )

sql_chatbot.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d7b8c40005ab285050.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 산출물

실행 후 생성되는 주요 파일:

- `lazada_shoppingmall.db`: SQLite 데이터베이스 파일
- Gradio Dashboard 링크
- Gradio Text-to-SQL 챗봇 링크

DB 파일 다운로드가 필요하면 아래 셀을 실행하세요.

In [23]:
# DB 파일 다운로드
try:
    from google.colab import files
    files.download(DB_PATH)
except Exception as e:
    print('Colab 환경이 아니거나 다운로드를 실행할 수 없습니다:', e)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>